# xLSTM — Multi-Source Cotton Yield Estimation (Türkiye)

This notebook trains and evaluates an **extended LSTM (xLSTM)** model
for cotton yield estimation from multivariate time-series (MTS) Earth
Observation data, following the experimental setup described in the
paper *"xLSTM for Multi-Source Cotton Yield Estimation and Temporal
Interpretability Across Agro-Ecological Regions in Türkiye"*
(Advances in Space Research).

xLSTM extends the standard LSTM with enhanced memory cells
(scalar **sLSTM** and matrix **mLSTM** variants) and exponential gating,
improving memory retention and gradient flow over long sequences.

**Input data (C = 20 variables / time step):**
- Sentinel-1 SAR backscatter (VV, VH)
- Sentinel-2 optical (EVI)
- ERA5-Land reanalysis (temperatures, water content, precipitation,
  evaporation, solar radiation, ...)
- SoilGrids static covariates (sand, silt, clay, bulk density, ...)

**Temporal setup:** bi-weekly (early/late) intervals over **June–October**,
sequence length **T = 10**.

**Target:** commune-level statistical yield (kg/da, TUIK).

**Pipeline of this notebook**
1. Environment & imports (incl. the `xlstm` package)
2. Data loading and indexing
3. Standardization (z-score)
4. MTS windowing -> `X (N, T, C)`, `y (N,)`
5. Train / validation / test split (**70 / 10 / 20**)
6. xLSTM model definition
7. Hyperparameter search reference (Optuna — see Table 2 of the paper)
8. Training with early stopping
9. Evaluation (R2, MAE, RMSE, MAPE)
10. Gradient-based temporal interpretability (saliency)


## 1. Environment & Imports

The experiments in the paper were run on an **NVIDIA T4 GPU (15 GB)**.
If you are running on Google Colab, mount Drive to access the dataset;
otherwise set `file_path` below to your local CSV.

The xLSTM blocks are provided by the `xlstm` package (pinned to the
version used in the paper). `ninja` is required to compile the CUDA
kernels.

In [ ]:
# (Colab only) mount Google Drive to access the dataset.
# Comment out if running locally.
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
except ModuleNotFoundError:
    print("Not running on Colab — skipping Drive mount.")

In [ ]:
!pip install xlstm==1.0.3
!pip install ninja

In [ ]:
import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.nn.init as init
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tqdm import tqdm

from xlstm import (
    xLSTMBlockStack,
    xLSTMBlockStackConfig,
    mLSTMBlockConfig,
    mLSTMLayerConfig,
    sLSTMBlockConfig,
    sLSTMLayerConfig,
    FeedForwardConfig,
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Load Data

The CSV holds one row per field per time step, with the multivariate
features and the `YIELD` target. Adjust `file_path` to point to the
dataset (publicly available at the project repository).

In [ ]:
file_path = '/content/gdrive/MyDrive/EsraHoca/aggregated_15_5_ege_az_ay.csv'  # set your path
data = pd.read_csv(file_path)
print("Raw shape:", data.shape)

In [ ]:
data.columns

In [ ]:
data.head(2)

In [ ]:
data["REGION_ID"].unique()  # 1: Aegean, 2: Mediterranean, 3: Southeastern Anatolia

Set a composite index so the feature matrix contains only the
predictor columns and the target. The index keeps field / period /
district / region / day identifiers available for later grouping.

In [ ]:
data.set_index(['FieldId', 'month_period', "DistrictName", "REGION_ID", "day"], inplace=True)
data.head(5)

In [ ]:
data.shape

## 3. Standardization

All variables (including the target `YIELD`) are z-score standardized
with a single `StandardScaler`. The scaler statistics for the `YIELD`
column are stored so predictions can be inverse-transformed back to
kg/da for reporting.

In [ ]:
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)
data_norm = pd.DataFrame(data_scaled, columns=data.columns, index=data.index)
data_norm.head(2)

## 4. Build Multivariate Time-Series Windows

Each field is converted into a non-overlapping window of length
`WINDOW_SIZE = 10` (the bi-weekly June–October sequence, T = 10). The
yield label is taken from the last step of each window. Output shapes:
`X = (N, T, C)` and `y = (N,)`.

In [ ]:
def df_to_X_y_non_overlap(df, window_size=5):
    df_as_np = np.array(df.drop(columns=["YIELD"]))
    df_yield_as_np = np.array(df["YIELD"])
    X = []
    y = []

    # Non-overlapping windows: step the loop by window_size.
    for i in range(0, len(df_as_np) - window_size + 1, window_size):
        row = [[a] for a in df_as_np[i:i+window_size]]
        X.append(row)
        label = df_yield_as_np[i+window_size-1]  # YIELD of the last step in the window
        y.append(label)

    return np.array(X).squeeze(), np.array(y)

In [ ]:
WINDOW_SIZE = 10
X, y = df_to_X_y_non_overlap(data_norm, WINDOW_SIZE)
print("X:", X.shape, "| y:", y.shape)

## 5. Dataset & DataLoader

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (torch.tensor(self.X[idx], dtype=torch.float32),
                torch.tensor(self.y[idx], dtype=torch.float32))

In [ ]:
batch_size = 32

## 6. Train / Validation / Test Split

Following the paper: **70% training, 10% validation, 20% test**
(`random_state=42`). The 20% test split is held out first; the
remaining 80% is split again so that validation is 10% of the total
(0.125 x 0.8 = 0.10).

In [ ]:
# 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)

# 10% validation of the total (0.125 of the remaining 80%)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.125, random_state=42)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

In [ ]:
train_dataset = TimeSeriesDataset(X_train, y_train)
val_dataset   = TimeSeriesDataset(X_val,   y_val)
test_dataset  = TimeSeriesDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

## 7. xLSTM Model

An input projection to an embedding space, followed by an
**xLSTM block stack** (mLSTM + sLSTM blocks with exponential gating),
a fully connected layer, a ReLU activation, and a linear regression
head. The block configuration (mLSTM/sLSTM layer settings, number of
heads, conv kernel size, `slstm_at`, feed-forward) follows the original
xLSTM design and is preserved as-is.

The model carries its own training and evaluation loops with:

- **Loss:** MSE
- **Optimizer:** Adam
- **Early stopping** on validation R2 (with `min_epochs` and `patience`)
- Logging of loss, gradient norms, and validation metrics per epoch

`inverse_transform_yield` maps standardized predictions back to kg/da
using the stored `YIELD` scaler statistics.

In [ ]:
import torch
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from xlstm import (
    xLSTMBlockStack,
    xLSTMBlockStackConfig,
    mLSTMBlockConfig,
    mLSTMLayerConfig,
    sLSTMBlockConfig,
    sLSTMLayerConfig,
    FeedForwardConfig,
)
import torch.nn.init as init

yield_index = list(data.columns).index('YIELD')

def inverse_transform_yield(yield_scaled):
    # İlk olarak, girdiyi numpy dizisine dönüştür
    yield_scaled = np.array(yield_scaled)
    # scaler.mean_ ve scaler.scale_ kullanarak YIELD sütununu tersine çevir
    yield_original = yield_scaled * scaler.scale_[yield_index] + scaler.mean_[yield_index]
    return yield_original

class xLSTMModel(nn.Module):
    @staticmethod
    def weight_init(m):
        if isinstance(m, nn.Linear) or isinstance(m, nn.Conv1d):
            init.kaiming_normal_(m.weight)
            if m.bias is not None:
                init.zeros_(m.bias)
        elif isinstance(m, nn.LSTM):
            for param in m.parameters():
                if len(param.shape) >= 2:
                    init.kaiming_normal_(param)
                else:
                    init.zeros_(param)

    def __init__(self, input_size=23, embedding_dim=128, context_length=10, num_blocks=2, fully_connected_size=16, lr = 0.0001):
        super(xLSTMModel, self).__init__()

        self.input_projection = nn.Linear(input_size, embedding_dim)
        self.context_length = context_length

        self.cfg = xLSTMBlockStackConfig(
            mlstm_block=mLSTMBlockConfig(
                mlstm=mLSTMLayerConfig(
                    conv1d_kernel_size=4, qkv_proj_blocksize=4, num_heads=4
                )
            ),
            slstm_block=sLSTMBlockConfig(
                slstm=sLSTMLayerConfig(
                    backend="vanilla",
                    num_heads=4,
                    conv1d_kernel_size=4,
                    bias_init="powerlaw_blockdependent",
                ),
                feedforward=FeedForwardConfig(proj_factor=1.3, act_fn="gelu"),
            ),
            context_length=self.context_length,
            num_blocks=num_blocks,
            embedding_dim=embedding_dim,
            slstm_at=[1],
        )

        self.xlstm_stack = xLSTMBlockStack(self.cfg)

        self.fc = nn.Linear(embedding_dim, fully_connected_size)
        self.relu = nn.ReLU()
        self.regressor = nn.Linear(fully_connected_size, 1)
        self.apply(self.weight_init)
        self.criterion = nn.MSELoss()
        self.optimizer = optim.Adam(self.parameters(), lr=lr)
        self.gradient_norms = []
        self.losses = []
        self.val_r2_log = []
        self.mae_log = []
        self.rmse_log = []
        self.mape_log = []

    def forward(self, x):
        batch_size = x.size(0)
        #print(f"Input shape: {x.shape}")

        x = self.input_projection(x)  # [batch, seq_len, embedding_dim]
        #print(f"Shape after input_projection: {x.shape}")

        if x is None:
            print("input_projection output is None!")
            return None

        x = self.xlstm_stack(x)
        #print(f"Shape after xlstm_stack: {x.shape}")

        if x is None:
            print("xlstm_stack output is None!")
            return None

        x = x[:, -1, :]  # Get the last time step for each original batch
        #print(f"Shape after slicing: {x.shape}")

        if x is None:
            print("Slicing output is None!")
            return None

        f_out = self.relu(self.fc(x))
       # print(f"Shape after fully connected layer: {f_out.shape}")

        if f_out is None:
            print("Fully connected layer output is None!")
            return None

        out = self.regressor(f_out)
        #print(f"Shape after regressor: {out.shape}")

        if out is None:
            print("Regressor output is None!")
            return None

        if torch.isnan(out).any():
            print("NaN values found in forward pass output.")

        return out  # Ensure we return the output without squeezing

    def train_model(self, train_loader, val_loader, test_loader=None, epochs=20, min_epochs=10, patience=5):
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.to(device)

        best_val_r2 = float('-inf')
        epochs_no_improve = 0
        loss_oscillations = []

        for epoch in range(epochs):
            self.train()
            running_loss = 0.0
            total_samples = 0
            epoch_gradient_norm = []

            for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}"):
                inputs = inputs.to(device, dtype=torch.float32)
                labels = labels.to(device, dtype=torch.float32)

                self.optimizer.zero_grad()
                outputs = self(inputs).squeeze()
                loss = self.criterion(outputs, labels)
                loss.backward()

                # Gradient Norm hesaplama
                total_norm = 0
                for p in self.parameters():
                    if p.grad is not None:
                        param_norm = p.grad.data.norm(2)
                        total_norm += param_norm.item() ** 2
                total_norm = total_norm ** 0.5
                epoch_gradient_norm.append(total_norm)

                self.optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                total_samples += inputs.size(0)

            # Epoch başına ortalama gradyan normu ekleme
            mean_gradient_norm = np.mean(epoch_gradient_norm)
            self.gradient_norms.append(mean_gradient_norm)

            epoch_loss = running_loss / total_samples
            self.losses.append(epoch_loss)

            # Loss osilasyonlarını hesaplama ve kaydetme
            if epoch > 0:
                loss_oscillation = abs(self.losses[-1] - self.losses[-2])
                loss_oscillations.append(loss_oscillation)

            # Validasyon setinde modeli test et ve metrikleri hesapla
            val_loss, mae, rmse, mape, val_r2 = self.test_model(val_loader, device, epoch, calculate_metrics=True)
            self.val_r2_log.append(val_r2)
            self.mae_log.append(mae)
            self.rmse_log.append(rmse)
            self.mape_log.append(mape)

            if val_r2 > best_val_r2:
                best_val_r2 = val_r2
                epochs_no_improve = 0
                torch.save(self.state_dict(), 'best_model.pth')
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= patience and epoch >= min_epochs:
                print("Early stopping triggered")
                break

            print(f'Epoch: {epoch + 1}, Loss: {epoch_loss:.4f}, Val R²: {val_r2:.4f}, MAE: {mae:.2f}, RMSE: {rmse:.2f}, MAPE: {mape:.2f}%')

        self.load_state_dict(torch.load('best_model.pth'))

        if test_loader:
            test_r2 = self.test_model(test_loader, device, epoch=epochs, calculate_metrics=False)
            print(f'Final Test R²: {test_r2:.4f}')

        return {
            "epoch_loss": self.losses,
            "r2": self.val_r2_log,
            "mae": self.mae_log,
            "rmse": self.rmse_log,
            "mape": self.mape_log,
            "gradient_norm": self.gradient_norms,
            "loss_oscillations": loss_oscillations
        }

    def test_model(self, loader, device, epoch, calculate_metrics=True):
        self.eval()
        running_loss = 0.0
        total_samples = 0
        predictions = []
        actuals = []

        with torch.no_grad():
            for inputs, labels in tqdm(loader, desc=f"Epoch {epoch + 1}", leave=False):
                inputs = inputs.to(device, dtype=torch.float32)
                labels = labels.to(device, dtype=torch.float32)

                outputs = self(inputs).squeeze()
                loss = self.criterion(outputs, labels)

                running_loss += loss.item() * inputs.size(0)
                total_samples += inputs.size(0)

                outputs = inverse_transform_yield(outputs.cpu().numpy())
                labels = inverse_transform_yield(labels.cpu().numpy())

                predictions.extend(outputs)
                actuals.extend(labels)

        loss = running_loss / total_samples
        predictions = np.array(predictions)
        actuals = np.array(actuals)

        if calculate_metrics:
            mae = mean_absolute_error(actuals, predictions)
            rmse = np.sqrt(mean_squared_error(actuals, predictions))
            mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100
            r2 = r2_score(actuals, predictions)
            return loss, mae, rmse, mape, r2
        else:
            r2 = r2_score(actuals, predictions)
            return r2

def inverse_transform_yield(yield_scaled):
    yield_scaled = np.array(yield_scaled)
    yield_original = yield_scaled * scaler.scale_[yield_index] + scaler.mean_[yield_index]
    return yield_original

## 8. Hyperparameter Search (Optuna) — Reference

Hyperparameters were tuned with **Optuna** using **5-fold
cross-validation**, maximizing the **Concordance Correlation
Coefficient (CCC)**. The search space and the optimal configuration
selected for xLSTM (Table 2 of the paper) are:

| Hyperparameter            | Tested range                         | Optimal |
|---------------------------|--------------------------------------|---------|
| Embedding dimension       | {16, 32, 64, 128, 256}               | **64**  |
| Number of blocks          | {2, 3, 4, 5, 6}                      | **2**   |
| Fully connected layer size| {4, 8, 16, 32, 64}                   | **64**  |
| Learning rate (Adam)      | [1e-5, 1e-1] (log-uniform)           | **6e-4**|

The search itself is **not re-run by default** (it is expensive). The
cell below reproduces the search procedure for reference; leave it
disabled (`RUN_OPTUNA = False`) and use the fixed optimal values in
the training section. The optimal values below are applied directly in
Section 9.

In [ ]:
RUN_OPTUNA = False  # set True to re-run the hyperparameter search (slow)

if RUN_OPTUNA:
    import optuna

    def concordance_correlation_coefficient(x, y):
        """Concordance Correlation Coefficient (CCC)."""
        if x.ndim == 1:
            x = x[:, np.newaxis]
        if y.ndim == 1:
            y = y[:, np.newaxis]
        sxy = np.sum(np.dot((x - x.mean())[:, 0], (y - y.mean())[:, 0])) / x.shape[0]
        rhoc = 2 * sxy / (np.var(x) + np.var(y) + (x.mean() - y.mean()) ** 2)
        return rhoc

    def objective(trial):
        embedding_dim = trial.suggest_categorical('embedding_dim', [16, 32, 64, 128, 256])
        num_blocks = trial.suggest_categorical('num_blocks', [2, 3, 4, 5, 6])
        lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
        fully_connected_size = trial.suggest_categorical('fully_connected_size', [4, 8, 16, 32, 64])

        kfold = KFold(n_splits=5, shuffle=True, random_state=42)
        scores = []
        for train_idx, test_idx in kfold.split(X_train):
            X_tr, X_va = X_train[train_idx], X_train[test_idx]
            y_tr, y_va = y_train[train_idx], y_train[test_idx]

            tr_loader = DataLoader(TimeSeriesDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
            va_loader = DataLoader(TimeSeriesDataset(X_va, y_va), batch_size=batch_size, shuffle=False)

            model = xLSTMModel(input_size=X_train.shape[2], embedding_dim=embedding_dim,
                               num_blocks=num_blocks, fully_connected_size=fully_connected_size,
                               lr=lr).to(device)
            model.train_model(tr_loader, va_loader, epochs=100, min_epochs=10, patience=5)

            preds = []
            model.eval()
            with torch.no_grad():
                for inputs, _ in va_loader:
                    preds.extend(model(inputs.to(device)).cpu().detach().numpy())
            scores.append(concordance_correlation_coefficient(y_va, np.array(preds)))
        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=10)
    print('Best trial:', study.best_trial.params)

## 9. Train the xLSTM (optimal configuration)

Trained with the Optuna-selected optimal hyperparameters from Table 2:
`embedding_dim=64`, `num_blocks=2`, `fully_connected_size=64`,
`lr=6e-4`. Early stopping monitors validation R2. The best checkpoint
(by validation R2) is saved to `best_model.pth` and reloaded at the
end of training.

In [ ]:
net = xLSTMModel(
    input_size=X_train.shape[2],
    embedding_dim=64,
    num_blocks=2,
    fully_connected_size=64,
    lr=6e-4,  # Optuna optimal
).to(device)
net.train()

metric = net.train_model(train_loader, val_loader, test_loader,
                         epochs=250, patience=25)

Optionally reload the best checkpoint explicitly (e.g. in a fresh
session) before evaluation.

In [ ]:
# net = xLSTMModel(input_size=X_train.shape[2], embedding_dim=64,
#                  num_blocks=2, fully_connected_size=64, lr=6e-4).to(device)
# net.load_state_dict(torch.load('best_model.pth'))
# net = net.to(device)

## 10. Evaluation on the Test Set

Predictions are inverse-transformed to kg/da before computing
**R2, MAE, RMSE, and MAPE**.

In [ ]:
y_pred, y_true = [], []

net.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = net(images).squeeze().cpu().numpy()
        y_pred.extend(outputs)
        y_true.extend(labels.cpu().numpy())

y_pred = inverse_transform_yield(np.array(y_pred))
y_true = inverse_transform_yield(np.array(y_true))

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
r2   = r2_score(y_true, y_pred)

print(f"R^2:  {r2:.3f}")
print(f"MAE:  {mae:.3f} kg/da")
print(f"RMSE: {rmse:.3f} kg/da")
print(f"MAPE: {mape:.3f} %")

### Predicted vs. actual yield

In [ ]:
errors = np.abs(y_true - y_pred)

plt.figure(figsize=(8, 8))
plt.scatter(y_true, y_pred, c=errors, cmap="jet", edgecolor='w', s=50)
plt.plot([min(y_true), max(y_true)], [min(y_true), max(y_true)],
         'black', linestyle='--')  # ideal line
plt.colorbar(label='Absolute error (kg/da)')
plt.xlabel('Actual yield (kg/da)')
plt.ylabel('Predicted yield (kg/da)')
plt.title('Predicted vs. Actual Cotton Yield (xLSTM)')
plt.tight_layout()
plt.show()

### Training curves

Per-epoch loss, validation R2, MAE, RMSE, MAPE, gradient norm, and loss
oscillations logged during training.

In [ ]:
epochs_updated = list(range(1, len(metric['epoch_loss']) + 1))

for metric_name, values in metric.items():
    plt.figure(figsize=(10, 5))
    if metric_name == 'loss_oscillations':
        epochs_range = list(range(2, len(values) + 2))
        plt.plot(epochs_range, values, marker='o', linestyle='-')
        last_epoch = epochs_range[-1] if values else 1
    else:
        plt.plot(epochs_updated, values, marker='o', linestyle='-')
        last_epoch = epochs_updated[-1]

    plt.title(metric_name.upper())
    plt.xlabel('Epoch')
    plt.ylabel(metric_name)
    plt.grid(True)

    if values:
        last_value = values[-1]
        plt.annotate(f'{last_value:.5f}',
                     xy=(last_epoch, last_value),
                     xytext=(last_epoch, last_value),
                     bbox=dict(boxstyle="round,pad=0.3", edgecolor='red', facecolor='white'))
    plt.tight_layout()
    plt.show()

## 11. Gradient-Based Temporal Interpretability

Interpretability uses **gradient-based saliency**: the gradient of the
predicted yield with respect to each input is computed on the test set
to reveal which **time intervals** and **features** the model relies on.

The temporal axis spans the study season from **early June (June_1)**
to **late October (October_2)**, consistent with the cotton phenology
described in the paper (vegetative/canopy development in June;
squaring/flowering and boll set in July–August; boll filling in
September; maturation/opening in October).

For xLSTM the paper highlights prominent importance around
**late August to early September** (late flowering / boll set and the
onset of boll filling). The saliency array is saved as `.npy` for the
cross-model comparison figures (LSTM vs. BiLSTM vs. xLSTM vs. Informer).

In [ ]:
# Compute input-gradient saliency on the test set.
test_input = torch.tensor(X_test, dtype=torch.float32).to(device)
test_input.requires_grad = True

net.zero_grad()
net.train()  # match training-mode behaviour used when computing saliency
outputs = net(test_input)
loss = outputs.mean()
loss.backward()

saliency = test_input.grad.data.cpu().numpy()   # (N, T, C)
mean_saliency = saliency.mean(axis=0)            # (T, C)

# Bi-weekly intervals, June-October (T = 10)
months = ["June_1", "June_2", "July_1", "July_2", "August_1",
          "August_2", "September_1", "September_2", "October_1", "October_2"]

# Saliency heatmap (time step x feature)
plt.figure(figsize=(12, 8))
plt.imshow(mean_saliency.T, aspect='auto', cmap='jet')
plt.colorbar(label='Saliency')
plt.xlabel('Time step')
plt.ylabel('Input feature')
plt.title('xLSTM Saliency Map (test set)')
plt.xticks(range(len(months)), months, rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Total saliency across time steps
plt.figure(figsize=(12, 6))
plt.plot(np.abs(mean_saliency).sum(axis=1), marker='o')
plt.xlabel('Time step')
plt.ylabel('Total saliency')
plt.title('xLSTM Total Saliency Across Time Steps')
plt.xticks(range(len(months)), months, rotation=45, ha='right')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Save saliency for the cross-model temporal-importance comparison figures.
np.save('saliency_xlstm_all_42.npy', saliency)